In [ ]:
import numpy as np
from fsvm_implement import fpca_smoothed_python, estimate_pc_scores, compute_gamma_automatic

# Test 1: BLUP scores should approximate direct projection when sigma2 → 0
np.random.seed(42)
n, J = 20, 100
Y = np.random.randn(n, J)

fpca = fpca_smoothed_python(Y, npc=3)

# Direct projection: (Y - mu) @ efunctions
Y_c = Y - fpca.mu
direct_scores = Y_c @ fpca.efunctions

# BLUP with very small sigma2
blup_scores = estimate_pc_scores(Y, fpca.mu, sigma2=1e-10, 
                                  evalues=fpca.evalues, efunctions=fpca.efunctions)

# They should be very close when sigma2 ≈ 0
diff = np.max(np.abs(direct_scores - blup_scores))
print(f'Test 1: BLUP ≈ direct projection when sigma2→0: max diff = {diff:.2e}')
assert diff < 1e-4, f'BLUP should match direct projection, got diff={diff}'
print('  PASSED')

# Test 2: BLUP scores should be shrunk toward zero when sigma2 is large
blup_shrunk = estimate_pc_scores(Y, fpca.mu, sigma2=1e6,
                                  evalues=fpca.evalues, efunctions=fpca.efunctions)
norm_shrunk = np.linalg.norm(blup_shrunk)
norm_direct = np.linalg.norm(direct_scores)
print(f'Test 2: Shrinkage with large sigma2: ||shrunk||={norm_shrunk:.4f} vs ||direct||={norm_direct:.4f}')
assert norm_shrunk < norm_direct * 0.01, 'Scores should be heavily shrunk'
print('  PASSED')

# Test 3: verify gamma computation matches kernlab logic
# For known data, check the quantile-based calculation
X = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [0.5, 0.5]])
from scipy.spatial.distance import pdist
dists_sq = pdist(X, 'sqeuclidean')
q10 = np.quantile(dists_sq, 0.1)
q90 = np.quantile(dists_sq, 0.9)
expected_gamma = np.mean([1.0/q90, 1.0/q10])
# compute_gamma_automatic uses subsampling, so test with full data
np.random.seed(0)
actual = compute_gamma_automatic(X)
print(f'Test 3: Gamma computation (may differ due to subsampling)')
print(f'  Expected (full data): {expected_gamma:.4f}')
print(f'  Actual (subsampled):  {actual:.4f}')
# Just verify it's a reasonable positive number
assert actual > 0, 'Gamma must be positive'
print('  PASSED (positive gamma)')

# Test 4: verify eigenvectors are orthonormal on grid
inner_products = fpca.efunctions.T @ fpca.efunctions  # (npc x npc)
off_diag = inner_products - np.eye(fpca.npc) * inner_products.diagonal()
max_off = np.max(np.abs(off_diag))
diag_vals = inner_products.diagonal()
print(f'Test 4: Eigenfunction orthogonality: max off-diagonal = {max_off:.2e}')
print(f'  Diagonal norms: {diag_vals}')
print('  PASSED' if max_off < 1e-10 else '  WARNING: not perfectly orthogonal')

# Test 5: eigenvalues should be positive and descending
print(f'Test 5: Eigenvalues positive and descending: {fpca.evalues}')
assert np.all(fpca.evalues > 0), 'All eigenvalues should be positive'
assert np.all(np.diff(fpca.evalues) <= 0), 'Eigenvalues should be descending'
print('  PASSED')

print()
print('All mathematical correctness tests passed.')



### Load data and create master df

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home/anushkasingh/Desktop/Thesis/Code/src"))
from load_data import read_data, create_combined_dataset
from baseline_correct import baseline_roy, process_all_samples
from best_params import fsvm_nested_cv, fsvm_nested_cv_XieOgden

path = ["../ALLDataGross/allKgData",
    "../ALLDataGross/BlindData",
    "../ALLDataGross/healthyCohort"
]
normVP = [[504, 425, 451, 454, 450, 474, 451, 471, 540, 467,
    550, 468, 481, 450, 515, 441, 452, 462, 453, 450, 452, 
    490, 504, 520, 525, 498, 542, 527, 550],
        [505, 503, 478, 453, 460, 494, 410, 413, 479, 489, 
    473, 464, 445, 499, 406, 455, 481, 388, 428, 466, 463, 
    520, 461],
    [420, 420, 428, 448, 417, 430, 420, 449, 483, 499, 
    438, 465, 438, 428, 503, 505, 504, 454, 515, 441, 
    404, 363]]
infoP = [["H", "PC", "PC", "H", "PC", "BC", "PC", "PC", "BC", "BC", 
    "PC", "PC", "PC", "PC", "H", "H","PC", "PC", "KC", "PC", "KC", 
    "PC", "BC", "BC", "PC", "KC", "PC", "PC", "PC"],
        ["H", "H" ,"H" ,"H", "KC", "KC", "PC", "PC" , "PC" ,"PC", 
        "PC", "PC", 'BC', 'KC', 'PC' , 'PC' , 'PC', 'KC' , 'PC', 'BC', 'PC', 'KC', 'BC'],
        ["F", "M", "M", "F", "F", "F", "F", "M", "M", "M", 
    "M", "F", "M", "M", "F", "M" ,"M", "M", "M", "M", 
    "M", "M" ]
]

df = create_combined_dataset(path,normVP,infoP)

df = create_combined_dataset(path,normVP,infoP)

df['intensity_baseline_corrected'] = None
for idx, row in df.iterrows():
    df.at[idx, "intensity_baseline_corrected"], _ ,_= baseline_roy(x=row["wavenumber"], y=row["intensity"], norm_factor_i=row['normVP'])
    print(f"{idx} done!")
print("baseline corrected!")

In [ ]:
# DEDUPLICATE
df = df.drop_duplicates(subset='original_filename', keep='first').reset_index(drop=True)

# Save the full 74-patient master df (all cohorts, before any filtering)
import os
pkl_path = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                        "..", "data_processed", "breath_data.pkl")
df.to_pickle(pkl_path)
print(f"Saved {len(df)}-row master df to {pkl_path}")
print(df["infoP"].value_counts())
print(df["category"].value_counts())

In [ ]:
df.head()

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

refund = importr("refund")
print("refund loaded successfully")

In [ ]:
df[df['original_filename'].isin(['20190405-02-001', '20190405-05-004', '20190606-07-015', '20190606-08-016'])].head(30)
# df[df['original_filename'].isin(['0190405-02-001'])]


# dubplicates: 
# 20190405-02-001            2
# 20190405-05-004            2
# 20190606-07-015            2
# 20190606-08-016            2

In [ ]:
df[df['original_filename'].duplicated(keep=False)].sort_values('original_filename')                                                                                

In [ ]:
df_blind = df[df["category"] == "blinddata"]
df = df[df["category"] != "blinddata"]
df.loc[df["infoP"].isin(["M", "F"]), "infoP"] = "H"
df_blind.head()

In [ ]:
df["infoP"].value_counts()

### FSVC on SR Windows — Replication of SVM Cases
Applies the same SR extraction + mean-center / std normalization as `SVM_notebook.ipynb`,
then runs joint nested CV (fsvc) and final LOOCV + 10×k-fold for the same four tasks.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.append('/home/anushkasingh/Desktop/Thesis/Code/classical_SVM_pipeline')
from sr_preprocessing import preprocess_all_srs
from fsvm_implement import (fsvc, fsvc_predict, run_fpca,
                             estimate_pc_scores, compute_gamma_automatic)
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
import numpy as np, pandas as pd

os.makedirs('eval_result_data', exist_ok=True)

# SR extraction + mean-center / std-normalize — identical to SVM_notebook cells 6-8
def apply_sr_preprocessing(dset):
    results = [
        preprocess_all_srs(np.array(dset['intensity_baseline_corrected'].iloc[i]),
                           np.array(dset['wavenumber'].iloc[i]))
        for i in range(len(dset))
    ]
    sr_names = list(results[0].keys())
    out = dset.copy()
    for sr in sr_names:
        out[f'{sr}_preprocessed'] = [results[i][sr]['spectrum'] for i in range(len(dset))]
    return out, sr_names

df_sr,       sr_names = apply_sr_preprocessing(df)
df_blind_sr, _        = apply_sr_preprocessing(df_blind)
SR_COLS = [f'{sr}_preprocessed' for sr in sr_names]
print(f"Training n={len(df_sr)}, Blind n={len(df_blind_sr)}")
print(f"SR windows: {sr_names}")


In [ ]:
# ── Task definitions + nested CV (fsvc joint grid search) ──────────────────
# 2 tasks; fsvc() finds best (tau, K, C) per config via CV.
# FSVC grids: smoothers + Cs from Xie & Ogden (2024); n_folds=9 matches SVM_notebook.

TASKS = {
    'H_vs_PC':    ['H', 'PC'],
    'H_vs_KC_BC_PC': ['H', 'PC', 'KC', 'BC'],
}

def build_X_y(dset, classes, sr_mode, sr_col):
    mask = np.isin(dset['infoP'], classes)
    d    = dset[mask].reset_index(drop=True) #filter to only patients relevant to this task
    y    = d['infoP'].values
    if len(classes) > 2:           # collapse cancer subtypes — same as SVM_notebook
        y = np.where(y == 'H', 'H', 'cancer')
    X = (np.array(list(d[sr_col])) if sr_mode == 'single'
         else np.hstack([np.array(list(d[c])) for c in SR_COLS]))
    return X, y

# or each of the 36 configs, run FSVC's joint cross-validation to find the best smoothing/complexity parameters, then store the fitted model and results.

# 8 single-SR × 2 tasks + 2 concat-all = 36 configs
configs = pd.DataFrame(
    [{'config_id': f'{sr.replace("_preprocessed", "")}__{task}',
      'sr_mode': 'single', 'sr_col': sr, 'task': task, 'classes': classes}
     for sr in SR_COLS for task, classes in TASKS.items()] +
    [{'config_id': f'concat_all__{task}',
      'sr_mode': 'concat', 'sr_col': 'all', 'task': task, 'classes': classes}
     for task, classes in TASKS.items()]
)
print(f"Total configs: {len(configs)}")

fsvc_models = {}   # config_id → FSVCResult
cv_rows     = []

for i, row in configs.iterrows():
    X, y   = build_X_y(df_sr, row['classes'], row['sr_mode'], row['sr_col'])
    counts = dict(zip(*np.unique(y, return_counts=True)))
    n_folds = min(9, min(counts.values()))   # clamp to min class count (same as SVM_notebook)

    print(f"\n[{i+1}/{len(configs)}] {row['config_id']}  "
          f"n={len(y)}, J={X.shape[1]}, folds={n_folds}  {counts}")

    model = fsvc(
        X, y, kernel='rbf',
        Ks=list(range(1, 11)),
        smoothers=[0.5, 1.0, 5.0, 10.0],
        Cs=[0.01, 0.2575, 0.505, 0.7525, 1.0],
        npc=min(10, X.shape[1] - 1),    # npc must be < J (num wavenumber grid points)
        n_folds=n_folds,
        stratified_folds=True, random_state=42,
    )
    fsvc_models[row['config_id']] = model
    cv_rows.append({
        'config_id': row['config_id'], 'task': row['task'],
        'sr_mode': row['sr_mode'],      'n_samples': len(y),
        'opt_tau': model.opt_tau,       'opt_K': model.opt_K,
        'opt_C':   model.opt_C,
        'cv_accuracy': round(model.cv_accuracy_matrix.max(), 4),
    })

cv_summary = pd.DataFrame(cv_rows)
cv_summary.to_csv('eval_result_data/fsvc_sr_best_params.csv', index=False)
print("\nNested CV summary:")
print(cv_summary[['config_id','task','opt_tau','opt_K','opt_C','cv_accuracy']].to_string())


In [ ]:
# ── Evaluation helpers ──────────────────────────────────────────────────────
from sklearn.metrics import matthews_corrcoef
from scipy.stats import beta as beta_dist

USE_R = False

def clopper_pearson(k, n, alpha=0.05):
    """Exact 95% CI for a binomial proportion (k successes out of n trials)."""
    lo = beta_dist.ppf(alpha/2,   k,   n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k)   if k < n else 1.0
    return round(float(lo), 3), round(float(hi), 3)

def _metrics(y_true, y_pred):
    """Returns acc, sens, spec, mcc, tp, tn, fp, fn.
    H always treated as negative class to avoid alphabetical-sort bugs."""
    acc  = float(np.mean(y_pred == y_true))
    uniq = np.unique(y_true)
    if 'H' in uniq:
        pos = [c for c in uniq if c != 'H'][0]
        cm  = confusion_matrix(y_true, y_pred, labels=['H', pos])
    else:
        cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens = tp/(tp+fn) if tp+fn > 0 else 0.
    spec = tn/(tn+fp) if tn+fp > 0 else 0.
    mcc  = matthews_corrcoef(y_true, y_pred)
    return acc, sens, spec, mcc, int(tp), int(tn), int(fp), int(fn)

def _fold_predict(X_tr, y_tr, X_te, model):
    """One CV fold: FPCA → BLUP scores → SVM with model's best (tau, K, C)."""
    npc  = min(model.opt_K + 2, X_tr.shape[0] - 1, X_tr.shape[1] - 1)
    fpca = run_fpca(X_tr, npc=npc, lam=model.opt_tau)
    # S_tr = (fpca.scores[:, :model.opt_K] if USE_R
    #         else estimate_pc_scores(X_tr, fpca.mu, fpca.sigma2,
    #                                 fpca.evalues, fpca.efunctions)[:, :model.opt_K])
    S_tr = estimate_pc_scores(X_tr, fpca.mu, fpca.sigma2,
                                    fpca.evalues, fpca.efunctions)[:, :model.opt_K]

    S_te = estimate_pc_scores(X_te, fpca.mu, fpca.sigma2,
                               fpca.evalues, fpca.efunctions)[:, :model.opt_K]
    gamma = compute_gamma_automatic(S_tr)
    svm   = SVC(kernel='rbf', C=model.opt_C, gamma=gamma)
    svm.fit(S_tr, y_tr)
    return svm.predict(S_te)

def fsvc_loocv(X, y, model):
    """LOOCV — each patient predicted exactly once; CIs are meaningful here."""
    y_pred = np.empty(len(y), dtype=object)
    for i in range(len(y)):
        mask = np.arange(len(y)) != i
        y_pred[i] = _fold_predict(X[mask], y[mask], X[[i]], model)[0]
    return _metrics(y, y_pred)   # (acc, sens, spec, mcc, tp, tn, fp, fn)

def fsvc_kfold(X, y, model, k=9, n_repeats=10):
    """10x stratified k-fold. MCC averaged over repeats; no CIs (not meaningful for k-fold)."""
    accs, sens, specs, mccs = [], [], [], []
    for rep in range(n_repeats):
        y_pred = np.empty(len(y), dtype=object)
        for tr, te in StratifiedKFold(k, shuffle=True, random_state=42+rep).split(X, y):
            y_pred[te] = _fold_predict(X[tr], y[tr], X[te], model)
        a, s1, s2, mcc, *_ = _metrics(y, y_pred)
        accs.append(a); sens.append(s1); specs.append(s2); mccs.append(mcc)
    return dict(accuracy=np.mean(accs),accuracy_std=np.std(accs),
                sensitivity=np.mean(sens), sensitivity_std=np.std(sens),
                specificity=np.mean(specs),specificity_std=np.std(specs),
                mcc=np.mean(mccs),         mcc_std=np.std(mccs),
                sens_ci_lo=None, sens_ci_hi=None,
                spec_ci_lo=None, spec_ci_hi=None,
                TP=None, TN=None, FP=None, FN=None)


In [ ]:
# ── Final evaluation: LOOCV + 10x k-fold (k=7, 9, 11) ────────────────────
# Exactly mirrors SVM_notebook Cell 29: K_VALUES=[7,9,11], n_repeats=10.
# k is clamped to min class count (same as SVM_notebook safe_folds logic).

K_VALUES  = [7, 9, 11]
eval_rows = []

for _, row in configs.iterrows():
    X, y   = build_X_y(df_sr, row['classes'], row['sr_mode'], row['sr_col'])
    model  = fsvc_models[row['config_id']]
    counts = dict(zip(*np.unique(y, return_counts=True)))
    n_neg  = int(np.sum(y == 'H'))          # n healthy
    n_pos  = len(y) - n_neg                  # n cancer
    base   = {'config_id': row['config_id'], 'task': row['task'],
              'sr_mode':   row['sr_mode'],   'n_samples': len(y),
              'n_pos': n_pos, 'n_neg': n_neg,
              'opt_tau':   model.opt_tau,    'opt_K': model.opt_K,
              'opt_C':     model.opt_C}

    print(f"LOOCV  {row['config_id']} ...")
    a, s1, s2, mcc, tp, tn, fp, fn = fsvc_loocv(X, y, model)
    sens_ci = clopper_pearson(tp, tp+fn)
    spec_ci = clopper_pearson(tn, tn+fp)
    eval_rows.append({**base, 'method': 'LOOCV', 'k_used': len(y),
                      'accuracy': a,  'accuracy_std': 0.,
                      'sensitivity': s1, 'sensitivity_std': 0.,
                      'sens_ci_lo': sens_ci[0], 'sens_ci_hi': sens_ci[1],
                      'specificity': s2, 'specificity_std': 0.,
                      'spec_ci_lo': spec_ci[0], 'spec_ci_hi': spec_ci[1],
                      'mcc': mcc, 'mcc_std': 0.,
                      'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

    for k in K_VALUES:
        safe_k = min(k, min(counts.values()))
        if safe_k < 2:
            print(f"  SKIP {k}-fold (min class={min(counts.values())})")
            continue
        if safe_k != k:
            print(f"  \u26a0 {k}-fold clamped to {safe_k}")
        print(f"  {k}-fold  {row['config_id']} ...")
        r = fsvc_kfold(X, y, model, k=safe_k, n_repeats=10)
        eval_rows.append({**base, 'method': f'{k}-fold (x10)', 'k_used': safe_k, **r})

results_df = pd.DataFrame(eval_rows)
results_df.to_csv('eval_result_data/fsvc_sr_evaluation_results.csv', index=False)

show = ['config_id','task','method','n_pos','n_neg','accuracy',
        'sensitivity','sens_ci_lo','sens_ci_hi',
        'specificity','spec_ci_lo','spec_ci_hi','mcc']
print("\nLOOCV results (sorted by accuracy):")
print(results_df[results_df['method']=='LOOCV'][show].sort_values('accuracy', ascending=False).to_string())


In [ ]:
results_df.columns

In [ ]:
results_df[                                                                                                                                                        
      (results_df['sensitivity'] >= 0.7) & (results_df['specificity'] >= 0.7)
  ].sort_values('accuracy', ascending=False).head(30)

In [ ]:
# ── Blind set evaluation ────────────────────────────────────────────────────
# Model is already refitted on full training data by fsvc() — no retraining needed.
# Matches SVM_notebook Cell 33 structure.

blind_rows = []

for _, row in configs.iterrows():
    classes    = row['classes']
    mask_blind = np.isin(df_blind_sr['infoP'], classes)
    df_bl      = df_blind_sr[mask_blind].reset_index(drop=True)
    if len(df_bl) == 0:
        continue

    y_blind = df_bl['infoP'].values
    if len(classes) > 2:
        y_blind = np.where(y_blind == 'H', 'H', 'cancer')

    X_blind = (np.array(list(df_bl[row['sr_col']])) if row['sr_mode'] == 'single'
               else np.hstack([np.array(list(df_bl[c])) for c in SR_COLS]))

    model  = fsvc_models[row['config_id']]
    y_pred = fsvc_predict(X_blind, model.fpca_result, model.svm_model, model.opt_K)
    a, s1, s2, mcc, tp, tn, fp, fn = _metrics(y_blind, y_pred)   # explicit labels — avoids sort-order bug

    blind_rows.append({
        'config_id': row['config_id'], 'task': row['task'],
        'sr_mode': row['sr_mode'],     'n_blind': len(y_blind),
        'accuracy': a, 'sensitivity': s1, 'specificity': s2,
        'opt_tau': model.opt_tau, 'opt_K': model.opt_K, 'opt_C': model.opt_C,
    })
    print(f"{row['config_id']:42s}: Acc={a:.3f}  Sens={s1:.3f}  Spec={s2:.3f}  n_blind={len(y_blind)}")

blind_df = pd.DataFrame(blind_rows)
blind_df.to_csv('eval_result_data/fsvc_sr_blind_results.csv', index=False)
print(f"\nSaved blind results for {len(blind_df)} configs.")


### Test on generated Data

In [ ]:
from genData import gen_fsvc_pca                                                                                                                                                                
from fsvm_implement import fsvc, evaluate_blind_test                                                                                                                                            
import numpy as np                                                                                                                                                                              
from scipy.special import legendre                                                                                                                                                              
                                                                                                                                                                                                
# ── Scenario 1 parameters (Xie & Ogden 2024, p.1184) ──────────────────────                                                                                                                    
np.random.seed(42)
J        = 100                          # time points                                                                                                                                           
k        = 5                            # number of true FPCs                                                                                                                                   
lambdas  = np.array([1.0, 0.6, 0.3, 0.1, 0.05])
grids    = np.linspace(0, 1, J)                                                                                                                                                                 
                                                                                                                                                                                                
# Orthonormal Legendre eigenfunctions on [0,1]: phi_d(t) = sqrt(2d+1)*P_d(2t-1)                                                                                                                 
eigenfunction = np.array([                                                                                                                                                                      
    np.sqrt(2*d + 1) * legendre(d)(2*grids - 1)                                                                                                                                                 
    for d in range(1, k + 1)                                                                                                                                                                    
])  # (k, J)                                                                                                                                                                                    
                                                                                                                                                                                                
# Three boundary functions (Settings 1-3)                                                                                                                                                       
settings = {
    "Setting 1 (linear)":      lambda a1, a2: a1 + 0.3 * a2,                                                                                                                                    
    "Setting 2 (interaction)": lambda a1, a2: a1 * a2 + a1**2 - 0.5,                                                                                                                            
    "Setting 3 (quadratic)":   lambda a1, a2: a1**2 + a2**2 - 0.9,                                                                                                                              
}                                                                                                                                                                                               
                                                                                                                                                                                                
# ── Run each setting ───────────────────────────────────────────────────────                                                                                                                   
for setting_name, bfun in settings.items():
    print(f"\n{'='*55}")                                                                                                                                                                        
    print(f"  {setting_name}")
    print(f"{'='*55}")                                                                                                                                                                          
                                                                                                                                                                                                
    # Generate train and test sets
    train = gen_fsvc_pca(n=50,   k=k, bfun=bfun, lambdas=lambdas,                                                                                                                               
                        grids=grids, eigenfunction=eigenfunction, noise_sigma=1.0)                                                                                                             
    test  = gen_fsvc_pca(n=1000, k=k, bfun=bfun, lambdas=lambdas,                                                                                                                               
                        grids=grids, eigenfunction=eigenfunction, noise_sigma=1.0)                                                                                                             
                                                                                                                                                                                                
    # discrete_data is (J, n) — transpose to (n, J) for fsvc                                                                                                                                    
    X_train = train["discrete_data"].T   # (50, 100)
    y_train = train["classlabel"]        # {-1, +1}                                                                                                                                             
                                                                                                                                                                                                
    X_test  = test["discrete_data"].T    # (1000, 100)                                                                                                                                          
    y_test  = test["classlabel"]                                                                                                                                                                
                                                                                                                                                                                                
    print(f"  Train: {X_train.shape},  class balance: {np.bincount(y_train == 1)}")                                                                                                             
    print(f"  Test:  {X_test.shape},   class balance: {np.bincount(y_test  == 1)}")                                                                                                             
                                                                                                                                                                                                
    # Fit FSVC (use_r=False — J=100 is small enough for Python fallback)                                                                                                                        
    model = fsvc(                                                                                                                                                                               
        X_train, y_train,                                                                                                                                                                       
        
        kernel="rbf",                                                                                                                                                                           
        smoothers=[1.0],                      # tau ignored by Python fallback
        Cs=[0.01, 0.2575, 0.505, 0.7525, 1.0],                                                                                                                                                  
        Ks=list(range(1, 6)),                                                                                                                                                                   
        npc=5,                                                                                                                                                                                  
        n_folds=5,                                                                                                                                                                              
        random_state=42,                                                                                                                                                                        
    )                                                                                                                                                                                           

    # Evaluate on test set                                                                                                                                                                      
    metrics = evaluate_blind_test(model, X_test, y_test)

    print(f"\n  Optimal: tau={model.opt_tau}, K={model.opt_K}, C={model.opt_C:.4f}")                                                                                                            
    print(f"  Test accuracy:   {metrics['accuracy']:.3f}")
    print(f"  Sensitivity:     {metrics['sensitivity']:.3f}  CI: {metrics['sensitivity_95ci']}")                                                                                                
    print(f"  Specificity:     {metrics['specificity']:.3f}  CI: {metrics['specificity_95ci']}")                                                                                                
    print(f"  AUC:             {metrics['auc']:.3f}")                                                                                                                                           
    print(f"  MCC:             {metrics['mcc']:.3f}")                                                                                                                                           
    print(f"  Confusion matrix:\n{metrics['confusion_matrix']}")                                                                                                                                
                

In [ ]:
from genData import gen_fsvc_pca
from fsvm_implement import fsvc, evaluate_blind_test                                                                                                                                            
import numpy as np                                                                                                                                                                              
from scipy.special import legendre
import matplotlib.pyplot as plt                                                                                                                                                                 
                
np.random.seed(0)                                                                                                                                                                               
J       = 100   
k       = 5                                                                                                                                                                                     
lambdas = np.array([1.0, 0.6, 0.3, 0.1, 0.05])
grids   = np.linspace(0, 1, J)                                                                                                                                                                  
eigenfunction = np.array([
    np.sqrt(2*d + 1) * legendre(d)(2*grids - 1)                                                                                                                                                 
    for d in range(1, k + 1)                                                                                                                                                                    
])
                                                                                                                                                                                                
bfun_s1 = lambda a1, a2: a1 + 0.3 * a2   # Setting 1                                                                                                                                            

n_reps       = 100                                                                                                                                                                              
accuracies_fsvc_rbf    = []
accuracies_fsvc_linear = []                                                                                                                                                                     
accuracies_svc_rbf     = []
accuracies_svc_linear  = []                                                                                                                                                                     
                
from sklearn.svm import SVC                                                                                                                                                                     
from sklearn.preprocessing import StandardScaler
                                                                                                                                                                                                
for rep in range(n_reps):
    # ── Generate data ──────────────────────────────────────────
    train = gen_fsvc_pca(50,   k, bfun_s1, lambdas, grids, eigenfunction, 1.0)                                                                                                                  
    test  = gen_fsvc_pca(1000, k, bfun_s1, lambdas, grids, eigenfunction, 1.0)                                                                                                                  
                                                                                                                                                                                                
    X_train = train["discrete_data"].T   # (50, 100)                                                                                                                                            
    y_train = train["classlabel"]                                                                                                                                                               
    X_test  = test["discrete_data"].T
    y_test  = test["classlabel"]                                                                                                                                                                

    # ── FSVC Gaussian ──────────────────────────────────────────                                                                                                                               
    m_rbf = fsvc(X_train, y_train, kernel="rbf",
                smoothers=[1.0], Cs=[0.01, 0.2575, 0.505, 0.7525, 1.0],                                                                                                                        
                Ks=list(range(1, 6)), npc=5, n_folds=5)                                                                                                                                        
    acc = evaluate_blind_test(m_rbf, X_test, y_test)["accuracy"]                                                                                                                                
    accuracies_fsvc_rbf.append(acc)                                                                                                                                                             
                
    # ── FSVC Linear ────────────────────────────────────────────                                                                                                                               
    m_lin = fsvc(X_train, y_train, kernel="linear",
                smoothers=[1.0], Cs=[0.01, 0.2575, 0.505, 0.7525, 1.0],                                                                                                                        
                Ks=list(range(1, 6)), npc=5, n_folds=5)                                                                                                                                        
    acc = evaluate_blind_test(m_lin, X_test, y_test)["accuracy"]                                                                                                                                
    accuracies_fsvc_linear.append(acc)                                                                                                                                                          
                                                                                                                                                                                                
    # ── Classical SVC (raw measurements) ───────────────────────                                                                                                                               
    for kernel, store in [("rbf", accuracies_svc_rbf),
                        ("linear", accuracies_svc_linear)]:                                                                                                                                   
        svc = SVC(kernel=kernel)
        svc.fit(X_train, y_train)                                                                                                                                                               
        store.append(np.mean(svc.predict(X_test) == y_test))
                                                                                                                                                                                                
    if (rep + 1) % 10 == 0:                                                                                                                                                                     
        print(f"Rep {rep+1}/100 done")
                                                                                                                                                                                                
# ── Boxplot matching Figure 1(a) layout ────────────────────────                                                                                                                               
fig, ax = plt.subplots(figsize=(8, 4))
data   = [accuracies_svc_linear, accuracies_svc_rbf,                                                                                                                                            
        accuracies_fsvc_linear, accuracies_fsvc_rbf]                                                                                                                                          
labels = ["SVC\nlinear", "SVC\nGaussian", "FSVC\nlinear", "FSVC\nGaussian"]                                                                                                                     
colors = ["#E41A1C", "#FF7F00", "#4DAF4A", "#377EB8"]                                                                                                                                           
                                                                                                                                                                                                
bp = ax.boxplot(data, patch_artist=True, widths=0.5)                                                                                                                                            
for patch, color in zip(bp["boxes"], colors):                                                                                                                                                   
    patch.set_facecolor(color)                                                                                                                                                                  
                
ax.set_xticklabels(labels)                                                                                                                                                                      
ax.set_ylabel("Accuracy")
ax.set_title("Scenario 1 Setting 1 — n=50, J=100, η~N(0,1)\n(replicate of Figure 1a centre group)")                                                                                             
ax.set_ylim(0.4, 1.0)                                                                                                                                                                           
plt.tight_layout()                                                                                                                                                                              
plt.savefig("fig1a_replication.png", dpi=150)                                                                                                                                                   
plt.show()                                                                                                                                                                                      
                                                                                                                                                                                                
print(f"\nMedian accuracy (100 reps):")
print(f"  SVC linear:    {np.median(accuracies_svc_linear):.3f}")                                                                                                                               
print(f"  SVC Gaussian:  {np.median(accuracies_svc_rbf):.3f}")                                                                                                                                  
print(f"  FSVC linear:   {np.median(accuracies_fsvc_linear):.3f}")
print(f"  FSVC Gaussian: {np.median(accuracies_fsvc_rbf):.3f}")                                                                                                                                 
                                                        

In [ ]:
import numpy as np                                                                                                                                                                              
import matplotlib.pyplot as plt                                                                                                                                                                 
import matplotlib.patches as mpatches                                                                                                                                                           
from sklearn.svm import SVC                                                                                                                                                                     
from genData import gen_dif_mean                                                                                                                                                              
from fsvm_implement import fsvc, evaluate_blind_test                                                                                                                                            
                                                                                                                                                                                                
# ── Mean functions (paper p.1185) ─────────────────────────────────────────                                                                                                                    
meanf1 = lambda t: np.cos(10*t - np.pi/4) + 0.5*np.sin(8*t  - np.pi/4)                                                                                                                          
meanf2 = lambda t: np.cos(8*t  - np.pi/4) + 0.5*np.sin(10*t - np.pi/4)                                                                                                                          
                                                                                                                                                                                                
# ── Simulation grid ────────────────────────────────────────────────────────                                                                                                                   
settings   = [(50, 40), (50, 100), (100, 50)]   # (N, J)                                                                                                                                        
noise_sds  = [1.0, np.sqrt(10)]                  # N(0,1) and N(0,10) variance                                                                                                                  
N_REPS     = 20    # paper uses 100; start with 20 to verify trend                                                                                                                              
N_TEST     = 1000  # paper uses 10000; 1000 sufficient to verify ordering                                                                                                                       
                                                                                                                                                                                                
# FSVC hyperparameter grids (paper values)                                                                                                                                                      
SMOOTHERS = [0.5, 1.0, 5.0, 10.0]                                                                                                                                                               
CS        = [0.01, 0.2575, 0.505, 0.7525, 1.0]                                                                                                                                                  
KS        = list(range(1, 6))                                                                                                                                                                   
                                                                                                                                                                                                
METHODS = ["SVC\nlinear", "SVC\nGaussian", "FSVC\nlinear", "FSVC\nGaussian"]                                                                                                                    
COLORS  = ["#E41A1C", "#FF7F00", "#4DAF4A", "#377EB8"]                                                                                                                                          
                                                                                                                                                                                                
# ── Run simulations ────────────────────────────────────────────────────────                                                                                                                 
# results[noise_idx][setting_idx][method_idx] = list of accuracies                                                                                                                              
results = [[[ [] for _ in METHODS] for _ in settings] for _ in noise_sds]                                                                                                                       

total = N_REPS * len(settings) * len(noise_sds)                                                                                                                                                 
done  = 0                                                                                                                                                                                     
                                                                                                                                                                                                
for n_idx, sd in enumerate(noise_sds):                                                                                                                                                          
    for s_idx, (N, J) in enumerate(settings):
        time = np.linspace(0, 1, J)                                                                                                                                                             
                                                                                                                                                                                                
        for rep in range(N_REPS):
            # Generate data                                                                                                                                                                     
            train = gen_dif_mean(N, time, meanf1, meanf2, sd)                                                                                                                                 
            test  = gen_dif_mean(N_TEST, time, meanf1, meanf2, sd)                                                                                                                              

            X_train = train["discrete_data"].T   # (N, J)                                                                                                                                       
            y_train = train["classlabel"]                                                                                                                                                     
            X_test  = test["discrete_data"].T                                                                                                                                                   
            y_test  = test["classlabel"]                                                                                                                                                        

            # ── SVC linear ────────────────────────────────────────────                                                                                                                        
            svc_lin = SVC(kernel="linear")                                                                                                                                                    
            svc_lin.fit(X_train, y_train)
            results[n_idx][s_idx][0].append(                                                                                                                                                    
                np.mean(svc_lin.predict(X_test) == y_test))
                                                                                                                                                                                                
            # ── SVC Gaussian ──────────────────────────────────────────                                                                                                                      
            svc_rbf = SVC(kernel="rbf")                                                                                                                                                         
            svc_rbf.fit(X_train, y_train)                                                                                                                                                     
            results[n_idx][s_idx][1].append(                                                                                                                                                    
                np.mean(svc_rbf.predict(X_test) == y_test))
                                                                                                                                                                                                
            # ── FSVC linear ───────────────────────────────────────────                                                                                                                        
            m_lin = fsvc(X_train, y_train, kernel="linear",
                        smoothers=SMOOTHERS, Cs=CS, Ks=KS,                                                                                                                                     
                        npc=5, n_folds=5, random_state=rep)                                                                                                                                  
            results[n_idx][s_idx][2].append(                                                                                                                                                    
                evaluate_blind_test(m_lin, X_test, y_test)["accuracy"])                                                                                                                       
                                                                                                                                                                                                
            # ── FSVC Gaussian ─────────────────────────────────────────                                                                                                                      
            m_rbf = fsvc(X_train, y_train, kernel="rbf",                                                                                                                            
                        smoothers=SMOOTHERS, Cs=CS, Ks=KS,                                                                                                                                     
                        npc=5, n_folds=5, random_state=rep)
            results[n_idx][s_idx][3].append(                                                                                                                                                    
                evaluate_blind_test(m_rbf, X_test, y_test)["accuracy"])                                                                                                                         

            done += 1                                                                                                                                                                           
            print(f"  [{done}/{total}]  sd={sd:.2f}  N={N} J={J}  rep={rep+1}")                                                                                                               
                                                                                                                                                                                                


In [ ]:
#── Plot — replicating Figure 2(a) layout ─────────────────────────────────                                                                                                                    
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)                                                                                                                                    
noise_labels = [r"$\eta_{ij} \sim N(0,1)$", r"$\eta_{ij} \sim N(0,10)$"]                                                                                                                        
x_labels     = [f"N={N},J={J}" for N, J in settings]                                                                                                                                            
                                                                                                                                                                                                
W       = 0.22  # box width                                                                                                                                                                    
OFFSETS = [-1.5*W, -0.5*W, 0.5*W, 1.5*W]   # 4 methods side by side per group                                                                                                                   
                                                                                                                                                                                                
for ax, n_idx, noise_lbl in zip(axes, range(2), noise_labels):                                                                                                                                
    for s_idx in range(len(settings)):                                                                                                                                                          
        x_center = s_idx + 1                                                                                                                                                                    
        for m_idx in range(len(METHODS)):
            data = results[n_idx][s_idx][m_idx]                                                                                                                                                 
            bp = ax.boxplot(                                                                                                                                                                  
                data,                                                                                                                                                                           
                positions=[x_center + OFFSETS[m_idx]],                                                                                                                                        
                widths=W,                                                                                                                                                                       
                patch_artist=True,
                medianprops=dict(color="black", linewidth=1.5),                                                                                                                                 
                whiskerprops=dict(linewidth=1),                                                                                                                                                 
                capprops=dict(linewidth=1),
                flierprops=dict(marker="o", markersize=3, alpha=0.5),                                                                                                                           
            )                                                                                                                                                                                   
            bp["boxes"][0].set_facecolor(COLORS[m_idx])
            bp["boxes"][0].set_alpha(0.8)                                                                                                                                                       
                                                                                                                                                                                                
    ax.set_xticks(range(1, len(settings) + 1))
    ax.set_xticklabels(x_labels, fontsize=10)                                                                                                                                                   
    ax.set_title(noise_lbl, fontsize=11)                                                                                                                                                        
    ax.set_ylabel("Accuracy" if n_idx == 0 else "")
    ax.set_ylim(0.6, 1.05)                                                                                                                                                                      
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)                                                                                                                     
    ax.grid(axis="y", alpha=0.3)                                                                                                                                                                
                                                                                                                                                                                                
# Legend                                                                                                                                                                                        
patches = [mpatches.Patch(color=c, label=m.replace("\n", " "))                                                                                                                                  
            for c, m in zip(COLORS, METHODS)]                                                                                                                                                    
fig.legend(handles=patches, loc="upper center", ncol=4,
            fontsize=9, bbox_to_anchor=(0.5, 1.02))                                                                                                                                              
                                                                                                                                                                                                
fig.suptitle("Scenario 3 (Setting 5): Different Mean Functions", y=1.06, fontsize=12)                                                                                                           
plt.tight_layout()                                                                                                                                                                              
plt.savefig("fig2a_scenario3_replication.png", dpi=150, bbox_inches="tight")                                                                                                                    
plt.show()                                                                                                                                                                                      

# ── Print median summary ───────────────────────────────────────────────────                                                                                                                   
print("\nMedian accuracy summary:")                                                                                                                                                           
for n_idx, sd in enumerate(noise_sds):                                                                                                                                                          
    print(f"\n  noise sd={sd:.2f}")
    for s_idx, (N, J) in enumerate(settings):                                                                                                                                                   
        print(f"    N={N} J={J}: ", end="")                                                                                                                                                   
        for m_idx, method in enumerate(METHODS):                                                                                                                                                
            med = np.median(results[n_idx][s_idx][m_idx])                                                                                                                                       
            print(f"{method.replace(chr(10),' ')}={med:.3f}  ", end="")
        print()                                                                                                                                                                                 
                                                                    